# 불균형 데이터 처리 (Imbalanced Data)
분류 문제에서 클래스별 데이터 개수가 크게 차이나는 경우를 불균형 데이터라고 한다.

예) 사기 탐지 데이터에서 정상 99% vs 사기 1%

## 왜 문제가 될까?
정상이 99%인 데이터에서 모델이 무조건 "정상"으로만 예측해도 정확도가 99%가 나온다.<br>
하지만 실제 우리가 잡고 싶은 것은 1%의 사기인데, 그건 전혀 못 잡는다.

→ **정확도(Accuracy) 외의 다른 지표**(Precision, Recall, F1)를 봐야 한다.

## 처리 방법

| 방법 | 설명 |
| --- | --- |
| 언더샘플링 | 다수 클래스의 데이터를 줄임 |
| 오버샘플링 | 소수 클래스의 데이터를 늘림 |
| SMOTE | 소수 클래스의 합성 데이터 생성 |
| 클래스 가중치 | 모델 학습 시 소수 클래스에 가중치 부여 |

## 1. 언더샘플링 (Under Sampling)
다수 클래스에서 일부만 무작위로 추출

In [ ]:
# pip install imbalanced-learn
from imblearn.under_sampling import RandomUnderSampler
import numpy as np

# 가상 데이터 생성
X = np.random.randn(1000, 4)
y = np.array([0]*900 + [1]*100)  # 불균형 (9:1)

rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X, y)

print(f"전: {np.bincount(y)}")
print(f"후: {np.bincount(y_res)}")

## 2. 오버샘플링 (Over Sampling)
소수 클래스 데이터를 복제

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

print(f"전: {np.bincount(y)}")
print(f"후: {np.bincount(y_res)}")

## 3. SMOTE
단순 복제가 아니라 소수 클래스 데이터 사이를 보간하여 새로운 합성 데이터를 만든다.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f"전: {np.bincount(y)}")
print(f"후: {np.bincount(y_res)}")

## 4. 클래스 가중치
데이터를 건드리지 않고, 모델 학습 시 소수 클래스에 더 큰 가중치를 부여

In [ ]:
from sklearn.linear_model import LogisticRegression

# class_weight="balanced"로 자동 가중치 부여
model = LogisticRegression(class_weight="balanced")

## 주의사항
**리샘플링은 학습 데이터에만 적용한다.** 테스트 데이터는 실제 분포를 유지해야 한다.

잘못된 예시:
```python
X_res, y_res = smote.fit_resample(X, y)  # 전체 데이터에 SMOTE
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res)  # ❌
```

올바른 예시:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
X_train, y_train = smote.fit_resample(X_train, y_train)  # ✓ 학습 데이터에만
```